# Finetune Qwen/Qwen2.5-3B-Instruct with LLaMA Factory

Please use a **free** Tesla T4 Colab GPU to run this!

Project homepage: https://github.com/hiyouga/LLaMA-Factory

## Install Dependencies

In [1]:
%cd /content/
%rm -rf LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
%ls
!pip install -e .[torch,bitsandbytes]

/content
Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 444, done.
remote: Counting objects: 100% (444/444), done.
remote: Compressing objects: 100% (338/338), done.
remote: Total 444 (delta 109), reused 325 (delta 88), pack-reused 0 (from 0)
Receiving objects: 100% (444/444), 5.14 MiB | 19.19 MiB/s, done.
Resolving deltas: 100% (109/109), done.
/content/LLaMA-Factory
assets/       docker/    Makefile        README.md         scripts/  tests/
CITATION.cff  examples/  MANIFEST.in     README_zh.md      setup.py  tests_v1/
data/         LICENSE    pyproject.toml  requirements.txt  src/
Obtaining file:///content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llamafactory (pyproject.toml) ... done
  Created wheel for llamafactory: filename=llamafactory-0.9.4.dev0-0.editable-py3-non

### Check GPU environment

In [2]:
import torch
try:
  assert torch.cuda.is_available() is True
except AssertionError:
  print("Please set up a GPU before using LLaMA Factory: https://medium.com/mlearning-ai/training-yolov4-on-google-colab-316f8fff99c6")

## Update Identity Dataset

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load model and tokenizer manually if needed
model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## Fine-tune model via LLaMA Board

In [4]:
!nvidia-smi


Wed Nov  5 02:28:42 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             54W /  400W |    6381MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [7]:
import json

args = {
  "stage": "sft",
  "do_train":True,
  "model_name_or_path": "Qwen/Qwen2.5-3B-Instruct", # Use the loaded model
  "dataset": "privacy_qa_7k_rel_32k",
  "template": "qwen", # Update template based on the model
  "finetuning_type": "lora",
  "lora_target": "all",
  "output_dir": "privacyqa_qwen_7k_rel_32k",
  "per_device_train_batch_size": 16,
  "gradient_accumulation_steps": 4,
  "lr_scheduler_type": "cosine",
  "logging_steps": 640,
  "warmup_ratio": 0.1,
  "save_steps": 500,
  "learning_rate": 5e-05,
  "num_train_epochs": 2.0,
  "max_samples": 32000,
  "max_grad_norm": 1.0,
  "loraplus_lr_ratio": 16.0,
  "fp16": True,
  "report_to": "none"                                      # disable wandb logging
}

json.dump(args, open("privacyqa_qwen_7k_rel_32k.json", "w", encoding="utf-8"), indent=2)
print("Config file created successfully!")
%cd /content/LLaMA-Factory/



Config file created successfully!
/content/LLaMA-Factory


## Fine-tune model via Command Line

It takes ~30min for training.

In [8]:
# Verify the config file was created correctly
config_path = "privacyqa_qwen_7k_rel_32k.json"
import os
print("Config file exists:", os.path.exists(config_path))

# Check the content
with open(config_path, "r") as f:
    saved_config = json.load(f)
print("Config keys:", list(saved_config.keys()))

Config file exists: True
Config keys: ['stage', 'do_train', 'model_name_or_path', 'dataset', 'template', 'finetuning_type', 'lora_target', 'output_dir', 'per_device_train_batch_size', 'gradient_accumulation_steps', 'lr_scheduler_type', 'logging_steps', 'warmup_ratio', 'save_steps', 'learning_rate', 'num_train_epochs', 'max_samples', 'max_grad_norm', 'loraplus_lr_ratio', 'fp16', 'report_to']


In [9]:
# Start the training with the config file
!llamafactory-cli train privacyqa_qwen_7k_rel_32k.json


2025-11-05 02:32:47.547575: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762309967.568689    3400 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762309967.575039    3400 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762309967.590475    3400 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1762309967.590501    3400 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1762309967.590503    3400 computation_placer.cc:177] computation placer alr

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"
adapter_path = "privacyqa_qwen_7k_rel_32k" # This should match your output_dir

# Load the base model
nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=nf4_config,
    device_map="auto",
    trust_remote_code=True
)

# Load the LoRA adapter
model = PeftModel.from_pretrained(model, adapter_path)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Prepare for inference
model.eval()

# Example inference
prompt = "<|im_start|>system\nYou are a legal expert, classifier that determines if a segment of text from a document is relevant to a user query.\n\nQuery: \"Can the application hide my photos without access privilege to my photos folder?\"\nSegment: \"D. Individual Rights\"\n\nAnswer whether the segment is RELEVANT or IRRELEVANT to the query. Answer only with \"RELEVANT\" or \"IRRELEVANT\".<|im_end|>\n<|im_start|>user\nQuery: \"How is my data protected?\"\nSegment: \"We implement a variety of security measures to maintain the safety of your personal information.\"\n\nAnswer whether the segment is RELEVANT or IRRELEVANT to the query. Answer only with \"RELEVANT\" or \"IRRELEVANT\".<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=50)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

system
You are a legal expert, classifier that determines if a segment of text from a document is relevant to a user query.

Query: "Can the application hide my photos without access privilege to my photos folder?"
Segment: "D. Individual Rights"

Answer whether the segment is RELEVANT or IRRELEVANT to the query. Answer only with "RELEVANT" or "IRRELEVANT".
user
Query: "How is my data protected?"
Segment: "We implement a variety of security measures to maintain the safety of your personal information."

Answer whether the segment is RELEVANT or IRRELEVANT to the query. Answer only with "RELEVANT" or "IRRELEVANT".
assistant
RELEVANT
